=============================================================
BOOKLY - Step 3: Feature Engineering & Selection
=============================================================
Input  : books_clean.csv (11,039 rows × 9 columns)
Output : X_train.csv, X_test.csv, y_train.csv, y_test.csv
         publisher_encoding.pkl, author_encoding.pkl,
         global_mean.pkl, feature_columns.json,
         plot_05_feature_importance.png
=============================================================

In [11]:
import pandas as pd
import numpy as np
import re
import json
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")

In [12]:
df = pd.read_csv('books_clean.csv')
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}\n")

Loaded: 11039 rows × 9 columns
Columns: ['title', 'authors', 'average_rating', 'language_code', 'num_pages', 'ratings_count', 'text_reviews_count', 'publisher', 'pub_year']



In [13]:
X = df.drop(columns=['average_rating'])
y = df['average_rating']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
)

print(f"Train: {len(X_train)} rows  |  Test: {len(X_test)} rows")
print(f"Train target mean: {y_train.mean():.4f}  |  Test: {y_test.mean():.4f}")
print()

GLOBAL_MEAN = float(y_train.mean())
print(f"Global mean (train only): {GLOBAL_MEAN:.4f}\n")

Train: 8831 rows  |  Test: 2208 rows
Train target mean: 3.9469  |  Test: 3.9288

Global mean (train only): 3.9469



In [14]:
def extract_title_features(title_series):
    """
    Extract 3 numeric signals from the raw title string.
    The regex r'\\(.*?#(\\d+)' finds the first occurrence of
    "(... #N)" in a title, the standard Goodreads series format.
    """
    is_series  = title_series.str.contains(r'\(.*?#\d+', regex=True).astype(int)

    # Extract the integer after # NaN for standalone books → fill with 0
    series_num = (
        title_series
        .str.extract(r'\(.*?#(\d+)')   # returns a 1-col DataFrame
        [0]                             # get the Series
        .astype(float)
        .fillna(0)
        .astype(int)
    )

    # Word count: a rough proxy for academic vs popular fiction
    # (academic titles tend to be longer)
    title_words = title_series.str.split().apply(len)

    return is_series, series_num, title_words


for split, X_split in [('train', X_train), ('test', X_test)]:
    is_series, series_num, title_words = extract_title_features(X_split['title'])
    X_split['is_series']   = is_series.values
    X_split['series_num']  = series_num.values
    X_split['title_words'] = title_words.values

X_train.drop(columns=['title'], inplace=True)
X_test.drop(columns=['title'],  inplace=True)

print("── Title features extracted ──")
print(f"  is_series: {X_train['is_series'].sum()} series books in train")
print(f"  series_num range: 0 - {X_train['series_num'].max()}")
print(f"  title_words range: {X_train['title_words'].min()} - {X_train['title_words'].max()}\n")


── Title features extracted ──
  is_series: 1881 series books in train
  series_num range: 0 - 2860
  title_words range: 1 - 36



In [15]:
# Extract num_authors from authors
for X_split in [X_train, X_test]:
    X_split['num_authors'] = X_split['authors'].str.split('/').apply(len)

print("── Author count extracted ──")
print(f"  num_authors: 1-{X_train['num_authors'].max()} co-authors\n")

── Author count extracted ──
  num_authors: 1-51 co-authors



In [16]:
for X_split in [X_train, X_test]:
    X_split['log_ratings_count']      = np.log1p(X_split['ratings_count'])
    X_split['log_text_reviews_count'] = np.log1p(X_split['text_reviews_count'])
    X_split.drop(columns=['ratings_count', 'text_reviews_count'], inplace=True)

print("── Log transforms applied ──")
print(f"  log_ratings_count:      {X_train['log_ratings_count'].min():.2f} – {X_train['log_ratings_count'].max():.2f}")
print(f"  log_text_reviews_count: {X_train['log_text_reviews_count'].min():.2f} – {X_train['log_text_reviews_count'].max():.2f}\n")

── Log transforms applied ──
  log_ratings_count:      0.69 – 14.74
  log_text_reviews_count: 0.00 – 11.37



In [17]:
# Target Encoding: publisher and authors
SMOOTHING = 10


def target_encode(train_series, y_train, global_mean, smoothing=10):
    """
    Compute smoothed target encoding from training data.

    Parameters
    ----------
    train_series : pd.Series  - category column in train set (e.g. publisher)
    y_train      : pd.Series  - target values aligned to train_series
    global_mean  : float      - overall mean of y_train (fallback)
    smoothing    : int        - higher = more shrinkage toward global mean

    Returns
    -------
    dict  {category_name: smoothed_mean}
    """
    df_temp = pd.DataFrame({'cat': train_series.values, 'target': y_train.values})
    stats   = df_temp.groupby('cat')['target'].agg(['mean', 'count'])

    smoothed = (
        (stats['count'] * stats['mean'] + smoothing * global_mean)
        /
        (stats['count'] + smoothing)
    )
    return smoothed.to_dict()

In [18]:
pub_encoding = target_encode(X_train['publisher'], y_train, GLOBAL_MEAN, SMOOTHING)

X_train['publisher_enc'] = (
    X_train['publisher'].map(pub_encoding).fillna(GLOBAL_MEAN)
)
X_test['publisher_enc'] = (
    X_test['publisher'].map(pub_encoding).fillna(GLOBAL_MEAN)   # unseen → global_mean
)

In [19]:
# Stats
unseen_pub = X_test['publisher'].apply(lambda x: x not in pub_encoding).sum()
print("── Publisher target encoding ──")
print(f"  Encoded {len(pub_encoding)} publishers from train set")
print(f"  Test rows with unseen publisher → global_mean fallback: {unseen_pub}")
print(f"  publisher_enc train range: {X_train['publisher_enc'].min():.3f} – {X_train['publisher_enc'].max():.3f}")

# Top 5 publishers by encoded value
top_enc = sorted(pub_encoding.items(), key=lambda x: -x[1])[:5]
bot_enc = sorted(pub_encoding.items(), key=lambda x:  x[1])[:5]
print("  Highest encoded publishers:", [(p, round(v, 3)) for p, v in top_enc])
print("  Lowest  encoded publishers:", [(p, round(v, 3)) for p, v in bot_enc])
print()

X_train.drop(columns=['publisher'], inplace=True)
X_test.drop(columns=['publisher'],  inplace=True)

── Publisher target encoding ──
  Encoded 2001 publishers from train set
  Test rows with unseen publisher → global_mean fallback: 264
  publisher_enc train range: 3.679 – 4.228
  Highest encoded publishers: [('VIZ Media', 4.228), ('東立', 4.218), ('VIZ Media LLC', 4.212), ('Library of America', 4.173), ('Andrews McMeel Publishing', 4.159)]
  Lowest  encoded publishers: [("Teacher's Pet Publications  Inc.", 3.679), ('Walter Foster Publishing', 3.679), ('Intercultural Publishing', 3.74), ('Cliffs Notes', 3.745), ('Harlequin Presents', 3.767)]



In [20]:
# Step 1+2: Build per-author encoding from training set
all_train_authors = (
    pd.DataFrame({
        'author': X_train['authors'].str.split('/').explode().str.strip(),
        'idx':    X_train['authors'].str.split('/').explode().index
    })
)
# Map each expanded row back to its rating
all_train_authors['target'] = y_train.loc[all_train_authors['idx']].values

stats_a = all_train_authors.groupby('author')['target'].agg(['mean', 'count'])
author_encoding = (
    ((stats_a['count'] * stats_a['mean'] + SMOOTHING * GLOBAL_MEAN)
     / (stats_a['count'] + SMOOTHING))
    .to_dict()
)

In [21]:
# Step 3: Average encodings for multi-author books
def encode_authors(author_str, enc_map, fallback):
    names = [a.strip() for a in str(author_str).split('/')]
    vals  = [enc_map.get(n, fallback) for n in names]
    return float(np.mean(vals))

X_train['author_enc'] = X_train['authors'].apply(
    encode_authors, enc_map=author_encoding, fallback=GLOBAL_MEAN
)
X_test['author_enc'] = X_test['authors'].apply(
    encode_authors, enc_map=author_encoding, fallback=GLOBAL_MEAN
)

unseen_auth = X_test['authors'].apply(
    lambda s: all(
        a.strip() not in author_encoding
        for a in s.split('/')
    )
).sum()
print("── Author target encoding ──")
print(f"  Encoded {len(author_encoding)} individual authors from train set")
print(f"  Test books where ALL authors are unseen → global_mean: {unseen_auth}")
print(f"  author_enc train range: {X_train['author_enc'].min():.3f} – {X_train['author_enc'].max():.3f}")

# Show Stephen King's encoding
sk_enc = author_encoding.get('Stephen King', GLOBAL_MEAN)
print(f"  Stephen King encoded as: {sk_enc:.4f} (appears in 99 books in dataset)")
print()

X_train.drop(columns=['authors'], inplace=True)
X_test.drop(columns=['authors'],  inplace=True)


── Author target encoding ──
  Encoded 7752 individual authors from train set
  Test books where ALL authors are unseen → global_mean: 480
  author_enc train range: 3.679 – 4.371
  Stephen King encoded as: 3.9886 (appears in 99 books in dataset)



In [22]:
# Encode language_code
X_train = pd.get_dummies(X_train, columns=['language_code'], drop_first=True, dtype=int)
X_test  = pd.get_dummies(X_test,  columns=['language_code'], drop_first=True, dtype=int)

# Align test to train columns (in case a language appeared in train but not test)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

lang_cols = [c for c in X_train.columns if c.startswith('language_code_')]
print("── language_code OHE ──")
print(f"  Added {len(lang_cols)} language binary columns: {lang_cols}")
print(f"  'eng' is the baseline (dropped) - all zeros = English\n")

── language_code OHE ──
  Added 7 language binary columns: ['language_code_fre', 'language_code_ger', 'language_code_jpn', 'language_code_mul', 'language_code_other', 'language_code_spa', 'language_code_zho']
  'eng' is the baseline (dropped) - all zeros = English



In [23]:
assert list(X_train.columns) == list(X_test.columns), \
    "Column mismatch between train and test!"

print("=" * 60)
print("FINAL FEATURE MATRIX")
print("=" * 60)
print(f"\nX_train: {X_train.shape}  |  X_test: {X_test.shape}")
print()
print("Feature list:")
for i, col in enumerate(X_train.columns, 1):
    print(f"  {i:2d}. {col}")
print()
print("X_train sample (first 3 rows):")
print(X_train.head(3).to_string())
print()
print("Null check — X_train:", X_train.isnull().sum().sum(),
      " | X_test:", X_test.isnull().sum().sum())

FINAL FEATURE MATRIX

X_train: (8831, 17)  |  X_test: (2208, 17)

Feature list:
   1. num_pages
   2. pub_year
   3. is_series
   4. series_num
   5. title_words
   6. num_authors
   7. log_ratings_count
   8. log_text_reviews_count
   9. publisher_enc
  10. author_enc
  11. language_code_fre
  12. language_code_ger
  13. language_code_jpn
  14. language_code_mul
  15. language_code_other
  16. language_code_spa
  17. language_code_zho

X_train sample (first 3 rows):
      num_pages  pub_year  is_series  series_num  title_words  num_authors  log_ratings_count  log_text_reviews_count  publisher_enc  author_enc  language_code_fre  language_code_ger  language_code_jpn  language_code_mul  language_code_other  language_code_spa  language_code_zho
6849        416      2006          0           0            6            2           4.442651                1.791759       3.942055    3.926561                  0                  0                  0                  0                    0       

In [24]:
print("\n── Running feature importance check (50-tree RF on train) ──")
rf_check = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
rf_check.fit(X_train, y_train)

importance = pd.Series(
    rf_check.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print("\nFeature importances (higher = more useful):")
for feat, imp in importance.items():
    bar = '█' * int(imp * 300)
    print(f"  {feat:40s} {imp:.4f}  {bar}")


# Plot
fig, ax = plt.subplots(figsize=(9, 6))
importance.sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title("Feature Importances — Random Forest (diagnostic)", fontweight='bold')
ax.set_xlabel("Mean decrease in impurity (importance score)")
ax.axvline(0.01, color='crimson', linestyle='--', linewidth=1, label='0.01 cutoff')
ax.legend()
plt.tight_layout()
plt.savefig('plot_05_feature_importance.png')
plt.close()
print("\n[Saved] plot_05_feature_importance.png")


── Running feature importance check (50-tree RF on train) ──

Feature importances (higher = more useful):
  author_enc                               0.5978  ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  publisher_enc                            0.1234  █████████████████████████████████████
  log_ratings_count                        0.0733  █████████████████████
  num_pages                                0.0701  █████████████████████
  log_text_reviews_count                   0.0368  ███████████
  pub_year                                 0.0346  ██████████
  title_words                              0.0324  █████████
  series_num                               0.0127  ███
  num_authors                              0.0108  ███
  is_series                                0.0043  █
  language_code_fre                        0.0015  
  language_code_spa       

In [25]:
# Conservative approach: keep everything above 0.005 importance.
# Features near 0 add noise and slow down training.
IMPORTANCE_CUTOFF = 0.005
weak_features = importance[importance < IMPORTANCE_CUTOFF].index.tolist()

print(f"\nFeatures below {IMPORTANCE_CUTOFF} importance cutoff: {weak_features}")
if weak_features:
    print(f"  → Dropping {len(weak_features)} weak features")
    X_train.drop(columns=weak_features, inplace=True, errors='ignore')
    X_test.drop( columns=weak_features, inplace=True, errors='ignore')
    print(f"  → Feature matrix reduced to {X_train.shape[1]} columns")
else:
    print("  → All features pass the cutoff, keeping all")

print(f"\nFinal features ({X_train.shape[1]}):")
for col in X_train.columns:
    print(f"  • {col}")


Features below 0.005 importance cutoff: ['is_series', 'language_code_fre', 'language_code_spa', 'language_code_ger', 'language_code_other', 'language_code_jpn', 'language_code_mul', 'language_code_zho']
  → Dropping 8 weak features
  → Feature matrix reduced to 9 columns

Final features (9):
  • num_pages
  • pub_year
  • series_num
  • title_words
  • num_authors
  • log_ratings_count
  • log_text_reviews_count
  • publisher_enc
  • author_enc


In [26]:
print("\n── Pearson correlations with average_rating (train) ──")
corrs = pd.concat([X_train, y_train], axis=1).corr()['average_rating'].drop('average_rating')
for feat, val in corrs.sort_values(ascending=False).items():
    direction = '↑' if val > 0 else '↓'
    print(f"  {direction} {feat:40s}  r = {val:+.4f}")


── Pearson correlations with average_rating (train) ──
  ↑ author_enc                                r = +0.6805
  ↑ publisher_enc                             r = +0.4549
  ↑ num_pages                                 r = +0.1609
  ↑ title_words                               r = +0.1387
  ↑ log_ratings_count                         r = +0.0853
  ↑ log_text_reviews_count                    r = +0.0316
  ↑ num_authors                               r = +0.0302
  ↓ series_num                                r = -0.0232
  ↓ pub_year                                  r = -0.0405


In [27]:
# CSV splits - for use in model training
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv( 'X_test.csv',  index=False)
y_train.to_csv('y_train.csv', index=False, header=True)
y_test.to_csv( 'y_test.csv',  index=False, header=True)

# Encoding maps for use in web app
# The web app needs these to transform a user's typed publisher name
# into the same numeric representation the model was trained on.
joblib.dump(pub_encoding,    'publisher_encoding.pkl')
joblib.dump(author_encoding, 'author_encoding.pkl')
joblib.dump(GLOBAL_MEAN,     'global_mean.pkl')

# Column list the web app must produce exactly these columns in this order
with open('feature_columns.json', 'w') as f:
    json.dump(list(X_train.columns), f, indent=2)

print("\n" + "=" * 60)
print("SAVED FILES")
print("=" * 60)
print("  X_train.csv, X_test.csv   — feature matrices")
print("  y_train.csv, y_test.csv   — target vectors")
print("  publisher_encoding.pkl    — smoothed publisher means (train)")
print("  author_encoding.pkl       — smoothed author means (train)")
print("  global_mean.pkl           — fallback for unseen categories")
print("  feature_columns.json      — column order for web app")
print("  plot_05_feature_importance.png")

print(f"""
FEATURE ENGINEERING SUMMARY
─────────────────────────────────────────────────────────
  Extracted from title   : is_series, series_num, title_words
  Extracted from authors : num_authors, author_enc (target enc.)
  Log-transformed        : ratings_count, text_reviews_count
  OHE (baseline=eng)     : language_code → {len(lang_cols)} binary columns
  Target encoded         : publisher → publisher_enc (smoothing=10)
  Target encoded         : authors   → author_enc   (smoothing=10)
  Columns dropped        : title, authors, publisher (raw text gone)
  Weak features removed  : {weak_features if weak_features else 'none'}
─────────────────────────────────────────────────────────
  Final X_train shape: {X_train.shape}
  Final X_test  shape: {X_test.shape}
  Ready for Model Training
""")



SAVED FILES
  X_train.csv, X_test.csv   — feature matrices
  y_train.csv, y_test.csv   — target vectors
  publisher_encoding.pkl    — smoothed publisher means (train)
  author_encoding.pkl       — smoothed author means (train)
  global_mean.pkl           — fallback for unseen categories
  feature_columns.json      — column order for web app
  plot_05_feature_importance.png

FEATURE ENGINEERING SUMMARY
─────────────────────────────────────────────────────────
  Extracted from title   : is_series, series_num, title_words
  Extracted from authors : num_authors, author_enc (target enc.)
  Log-transformed        : ratings_count, text_reviews_count
  OHE (baseline=eng)     : language_code → 7 binary columns
  Target encoded         : publisher → publisher_enc (smoothing=10)
  Target encoded         : authors   → author_enc   (smoothing=10)
  Columns dropped        : title, authors, publisher (raw text gone)
  Weak features removed  : ['is_series', 'language_code_fre', 'language_code_spa', '